# Odd customers

Eight synthetic customers are appended to the Day 2 table. They are not in the CSV. Four of them spent far too much for one order. Four placed many orders and spent almost nothing. Seed 42 sets how far out they sit.

The check uses three June columns only: days since the last order, orders in 90 days, and spend in 90 days. The scale is fit on the real rows, then applied to everyone. The July columns stay out.

A z-score rule flags the rows with the largest absolute scaled value. `IsolationForest` in scikit-learn 1.9 scores with `decision_function`. Lower is more anomalous. With `contamination="auto"`, a negative score is the default cutoff. That cutoff flags many ordinary rows on this table, so do not use it as the answer. Count how many of the 8 planted rows fall inside the top N flags.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
root = None
for candidate in [here, *here.parents]:
    marker = candidate / "lib" / "leaks.py"
    if marker.is_file() and (candidate / "app.py").is_file():
        root = candidate
        break
if root is None:
    raise FileNotFoundError("Start Jupyter from 02-unsupervised-learning.")
lib = root / "lib"
if str(lib) not in sys.path:
    sys.path.insert(0, str(lib))
ROOT = root
print("module", ROOT)


In [ ]:
from leaks import built_table
from anomalies import forest_decision, planted_recall, top_flags, with_planted, z_strength

frame = with_planted(built_table())
print("rows", len(frame), "planted", int(frame["planted_anomaly"].sum()))


In [ ]:
top_n = None  # how many rows to flag

if top_n is None:
    raise NotImplementedError("Set top_n. Try 16.")


In [ ]:
forest = forest_decision(frame)
strength = z_strength(frame)
print("isolation forest recall", round(planted_recall(frame, top_flags(forest, top_n, low_is_anomaly=True)), 3))
print("z-score recall", round(planted_recall(frame, top_flags(strength, top_n, low_is_anomaly=False)), 3))
print("forest top 8 recall", round(planted_recall(frame, top_flags(forest, 8, low_is_anomaly=True)), 3))
print("z-score top 8 recall", round(planted_recall(frame, top_flags(strength, 8, low_is_anomaly=False)), 3))


Write one sentence: in the top 8 flags, which rule catches more of the planted rows?